In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

In [ ]:
# SECTION 2: LOAD DATASET

df = pd.read_csv("Churn_Modelling.csv")

print("First 5 Records:")
display(df.head())

print("\nDataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

In [ ]:
# SECTION 3: DATA EXPLORATION

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistical Summary:")
display(df.describe())

print("\nChurn Distribution:")
print(df['Exited'].value_counts())

# Visualize churn distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Exited')
plt.title("Bank Customer Churn Distribution")
plt.xlabel("Exited (0 = No, 1 = Yes)")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
# SECTION 4: DATA PREPROCESSING

# Remove duplicate records
df = df.drop_duplicates()

# Remove missing values
df = df.dropna()

# Drop unnecessary columns
df = df.drop(
    columns=['RowNumber', 'CustomerId', 'Surname'],
    errors='ignore'
)

# Separate features and target
X = df.drop('Exited', axis=1)
y = df['Exited'].astype(int)

print("\nFeatures:")
display(X.head())

print("\nTarget Values:")
print(y.value_counts())

In [ ]:
# SECTION 5: ENCODE CATEGORICAL DATA AND NORMALIZE

# Identify numerical and categorical columns
categorical_cols = X.select_dtypes(
    include=['object']
).columns.tolist()

numerical_cols = X.select_dtypes(
    exclude=['object']
).columns.tolist()

print("Categorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore',
                              drop='first'), categorical_cols)
    ]
)


In [ ]:
# SECTION 6: SPLIT DATASET INTO TRAINING AND TESTING

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)


In [ ]:
# SECTION 7: BUILD NEURAL NETWORK MODEL

# Create neural network using MLPClassifier
nn_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=300,
    random_state=42,
    early_stopping=True
)

# Create complete pipeline
model = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', nn_model)
])

# Train model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

print("\nNeural Network Model Trained Successfully!")

In [ ]:
# SECTION 8: OPTIMIZE NEURAL NETWORK MODEL

# Define hyperparameters
param_grid = {
    'classifier__hidden_layer_sizes': [
        (32,),
        (64, 32),
        (128, 64)
    ],
    'classifier__alpha': [
        0.0001,
        0.001
    ],
    'classifier__learning_rate_init': [
        0.001,
        0.01
    ]
}

# Grid Search
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

# Train and optimize
grid_search.fit(X_train, y_train)

# Best parameters
print("\nBest Parameters:")
print(grid_search.best_params_)

print("\nBest Cross Validation Accuracy:")
print(grid_search.best_score_)

In [ ]:
# SECTION 9: PREDICTION USING OPTIMIZED MODEL

best_model = grid_search.best_estimator_

# Predict test data
y_pred = best_model.predict(X_test)

print("\nPredicted Values:")
print(y_pred[:20])

print("\nActual Values:")
print(y_test.iloc[:20].values)


# SECTION 10: MODEL EVALUATION USING ACCURACY

accuracy = accuracy_score(y_test, y_pred)

print("\n========== MODEL ACCURACY ==========")
print("Neural Network Accuracy:", round(accuracy * 100, 2), "%")


In [ ]:
# SECTION 11: CONFUSION MATRIX

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Churned', 'Churned'],
    yticklabels=['Not Churned', 'Churned']
)

plt.title("Neural Network Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()


In [ ]:

# SECTION 12: CLASSIFICATION REPORT

print("\n========== CLASSIFICATION REPORT ==========")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=['Not Churned', 'Churned']
    )
)


# SECTION 13: FINAL CONCLUSION

print("\n========== FINAL CONCLUSION ==========")

print("Bank Customer Churn Prediction completed successfully.")

print("Best Neural Network Parameters:")
print(grid_search.best_params_)

print("\nFinal Model Accuracy:",
      round(accuracy * 100, 2), "%")

print("\nConfusion Matrix:")
print(cm)

print("\nPractical Completed Successfully!")
